# Cane Corso Growth Intelligence

## Project Concept and Mathematical Framing

This notebook explains the main idea of the project before the lecture-specific notebooks.

The goal is to show that the project is not only a weight prediction exercise. It is a mathematical growth-profiling system for predictive monitoring and early growth pattern detection.

The project is educational and does not provide veterinary diagnosis, medical advice, pedigree proof or breed certification.

In [ ]:
from pathlib import Path
import sys


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Project root not found. Open this notebook from the project folder or adjust the working directory.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Notebook ready. Project root: {PROJECT_ROOT}")


## 1. Product Idea

A Cane Corso owner can record simple information over time:

- age;
- weight;
- sex;
- height or other body measurements;
- repeated measurements over time.

The system can transform those records into machine-learning insights:

```text
owner record -> feature vector -> model prediction -> metric -> explanation -> monitoring signal
```

A useful output should not be only a number. It should explain expected growth, deviation, confidence and limitations.

## 2. Why Growth Monitoring Matters for Large Breeds

Large and giant breed puppies can grow quickly, but faster growth is not automatically better growth. If weight gain is too rapid or the growth pattern is uncontrolled, the developing bones and joints may be placed under extra mechanical stress.

This gives the project a practical motivation: growth data should not be treated only as numbers to predict. The more interesting task is to model the development trajectory and compare each record with an expected pattern.

In this project, the machine-learning output is a monitoring signal, not a medical conclusion. It can help describe whether a record appears close to the learned pattern, unusually high, unusually low or worth closer observation. Veterinary diagnosis and health decisions remain outside the scope of the model.

```text
growth record -> mathematical profile -> deviation / probability / cluster -> responsible interpretation
```

## Standard Mathematical Formulation Used in Every Notebook

Every notebook in this project follows the same mathematical structure. This keeps the work aligned with the course requirement for analysis, mathematics, and Python implementation.

### Input vector `X`

Each model receives a numerical or encoded representation of a growth record:

```text
X = [age, weight, sex_or_gender, body_measurements, engineered_growth_features]
```

The exact columns change by notebook, but the principle is always the same: real-world dog growth information is converted into a feature vector.

### Target `y`

The target depends on the task:

```text
Regression:      y = weight_kg or expected future weight
Classification:  y = growth_status_binary
Clustering:      no known y; the model discovers groups
Time series:     y_t = value at time t
```

### Model function `f(x)`

The model learns a mapping from input features to an output:

```text
Regression:      f(x) ≈ expected_weight
Classification:  f(x) = P(needs_attention | x)
Clustering:      f(x) = cluster_id
```

### Loss function

Training means choosing model parameters that reduce error:

```text
Regression loss:      MSE = mean((y_real - y_pred)^2)
Classification loss:  LogLoss / Cross-Entropy
Clustering objective: within-cluster distance or density separation
```

### Metrics

Models are evaluated with metrics appropriate for the task:

```text
Regression:      MAE, MSE, RMSE, R²
Classification:  Accuracy, Precision, Recall, F1, ROC-AUC
Clustering:      Inertia, Silhouette Score, cluster interpretation
```

### Interpretation

The output is translated into an owner-friendly growth-monitoring signal, not a medical diagnosis.

### Limitations

The project uses public dog growth data and educational samples. Results depend on data quality, feature selection, assumptions, and responsible interpretation.


## 2. Mathematical Feature Vector

Machine learning models do not understand the dog directly. They learn from a numerical representation.

A growth record can be represented as:

```text
x = [age_months, weight_kg, height_cm, sex_encoded, body_ratio, growth_velocity, deviation_from_expected]
```

This vector is the bridge between the real-world problem and the mathematical problem.

In [1]:
import numpy as np

feature_names = [
    "age_months",
    "weight_kg",
    "height_cm",
    "sex_encoded",
    "body_ratio",
    "growth_velocity",
    "deviation_from_expected",
]

# Example owner record converted into a mathematical vector.
age_months = 5.0
weight_kg = 28.0
height_cm = 52.0
sex_encoded = 1.0  # example: 1 = male, 0 = female
previous_weight_kg = 23.5
previous_age_months = 4.0
predicted_expected_weight_kg = 26.5

body_ratio = weight_kg / height_cm
growth_velocity = (weight_kg - previous_weight_kg) / (age_months - previous_age_months)
deviation_from_expected = weight_kg - predicted_expected_weight_kg

x = np.array([
    age_months,
    weight_kg,
    height_cm,
    sex_encoded,
    body_ratio,
    growth_velocity,
    deviation_from_expected,
])

for name, value in zip(feature_names, x):
    print(f"{name}: {value:.3f}")

age_months: 5.000
weight_kg: 28.000
height_cm: 52.000
sex_encoded: 1.000
body_ratio: 0.538
growth_velocity: 4.500
deviation_from_expected: 1.500


## 3. Regression Task

Regression learns a function:

```text
y = f(x)
```

In this project, the regression model can estimate expected bodyweight or a growth trend.

The model error is measured through residuals:

```text
residual = real_weight - predicted_weight
```

The training objective for ordinary least squares is:

```text
minimize sum((y_real - y_pred)^2)
```

In [2]:
# Simple residual and squared-error example
real_weight = 28.0
predicted_weight = 26.5

residual = real_weight - predicted_weight
squared_error = residual ** 2

print(f"Residual: {residual:.2f} kg")
print(f"Squared error: {squared_error:.2f}")

Residual: 1.50 kg
Squared error: 2.25


## 4. Classification Task

Classification learns a probability or class.

For this project:

```text
P(needs_attention | x)
```

The model output should be interpreted as a growth-monitoring signal, not as a diagnosis.

A threshold converts probability into a label:

```text
if probability >= threshold -> needs_attention
else -> normal_growth
```

In [3]:
import math

def sigmoid(z: float) -> float:
    return 1 / (1 + math.exp(-z))

# Example score produced by a logistic regression model.
z = 0.85
probability_needs_attention = sigmoid(z)
threshold = 0.5

predicted_label = "needs_attention" if probability_needs_attention >= threshold else "normal_growth"

print(f"P(needs_attention | x): {probability_needs_attention:.3f}")
print(f"Threshold: {threshold:.2f}")
print(f"Predicted label: {predicted_label}")

P(needs_attention | x): 0.701
Threshold: 0.50
Predicted label: needs_attention


## 5. Clustering Task

Clustering will be used to search for natural growth-pattern groups without known labels.

The idea is:

```text
nearby records in feature space -> similar growth pattern
```

Possible cluster interpretations could be:

- steady growth;
- fast early growth;
- slower development;
- irregular pattern.

This stage makes the project more interesting because it asks what structure exists in the data before labels are defined.

## 6. Data Source Honesty

The project uses two data layers:

1. a small educational Cane Corso-style prototype sample;
2. a real public dog growth dataset from University of Liverpool DataCat.

The project does not claim to have private Cane Corso veterinary records.

The correct statement is:

```text
The project uses public dog growth data as a foundation and applies it to a Cane Corso-oriented growth intelligence concept.
```

## 7. Responsible Interpretation

Correct interpretation:

```text
The model gives an educational growth-monitoring signal based on available data.
```

Incorrect interpretation:

```text
The model diagnoses health problems or replaces a veterinarian.
```

This boundary keeps the project useful, safe and academically honest.

## 8. Final Project Strategy

The project should be presented as one coherent mathematical story:

```text
real-world owner problem
-> mathematical feature vector
-> regression for expected growth
-> classification for growth signal
-> clustering for unknown profiles
-> time series for trajectory
-> dimensionality reduction for visualization
-> MLflow for experiment tracking
```

This makes the project more advanced than a basic weight-prediction assignment.